# Reporte comparativo de IA para Texas Hold'em

Este notebook ahora funciona como capa de presentación para el backend importable `poker_ai`. Las reglas del motor, el comportamiento de agentes, las ayudas de entrenamiento y los cálculos de ROI viven en `src/poker_ai/` y se verifican con pytest.

## Ruta de verificación

Usá `uv sync --extra dev` para instalar las herramientas locales sin modificar Python global. Luego ejecutá `uv run pytest` y, cuando se necesite validar el notebook, `uv run jupyter nbconvert --to notebook --execute poker_ia_comparativa_final.ipynb --output poker_ia_comparativa_final.executed.ipynb`.

## TD entrenado y persistente

La versión anterior de `TDLearning` podía quedar peor que `Random` porque el reporte evaluaba una tabla Q vacía y el desempate de valores cero podía seleccionar `fold`. El flujo actual evita `fold` en estados no vistos cuando existe `check`/`call`, actualiza Q con la recompensa terminal de la mano y agrega un bucket preflop (`premium`, `strong`, `playable`, `weak`) calculado solo con las dos cartas privadas del jugador. Esa señal es leakage-safe: no usa board, cartas del oponente, historial futuro ni resultado de la mano.

Para regenerarlo: `uv run python -m poker_ai.evaluation.train_td` (α=0.1, 200 manos, semilla 17). Por defecto el reporte usa `POKER_REPORT_FRESH_TD=1` para entrenar un snapshot fresco en memoria y no depender de métricas stale de `artifacts/td_qtable.json`. El entrenamiento separa RNG de mazo, exploración TD y oponente desde una semilla maestra.

La compuerta `td_meets_gate` exige ROI > -1.0 contra `Random` y `Call` con semillas fijas `(101, 202, 303)`. El reporte ahora compara `EnsembleNoTD` contra `EnsembleWithTD` como filas separadas: el primero usa `Minimax`/`Bayesian`/`Markov` con pesos `0.5/0.3/0.2`, y el segundo agrega TD con pesos `0.45/0.27/0.18/0.10`.

In [1]:
import pandas as pd
from IPython.display import Markdown, display

from poker_ai.evaluation import config_from_env, render_spanish_conclusions, run_report_experiment


In [2]:
# Valores por defecto: humo/CI rápido. Para corridas académicas:
# POKER_REPORT_HANDS=200 POKER_REPORT_SEEDS=42,43,44,45,46 uv run jupyter nbconvert --to notebook --execute poker_ia_comparativa_final.ipynb --output poker_ia_comparativa_final.executed.ipynb
config = config_from_env()
config


ExperimentConfig(hands_per_seed=100, seeds=(42, 43, 44, 45, 46), starting_stack=1000, max_raises_per_street=4)

## Comparación modular reducida

La configuración por defecto mantiene `POKER_REPORT_HANDS` y `POKER_REPORT_SEEDS` en valores pequeños para que el notebook sea práctico en verificación local y CI. Para una presentación académica, aumentá ambas variables de entorno y reportá los parámetros exactos. La tabla incluye `EnsembleNoTD` y `EnsembleWithTD` con composición y pesos visibles para comparar si la señal TD ayuda o arrastra la votación.

In [3]:
results = run_report_experiment(config)
summary = pd.DataFrame(
    [
        {
            "Agente": row.agent_name,
            "Oponente": row.opponent_name,
            "Semillas": row.seeds,
            "Manos por semilla": row.hands_per_seed,
            "Manos totales": row.total_hands,
            "Tasa de victoria": row.win_rate,
            "Ganancia total": row.total_profit,
            "Inversión total": row.total_invested,
            "ROI": row.roi,
            "Ganancia promedio": row.avg_profit,
            "Decisión prom. ms": row.avg_decision_ms,
            "Composición": ", ".join(row.composition),
            "Pesos": ", ".join(str(weight) for weight in row.weights),
        }
        for row in results
    ]
).sort_values(["ROI", "Ganancia promedio"], ascending=False)
summary


,Agente,Oponente,Semillas,Manos por semilla,Manos totales,Tasa de victoria,Ganancia total,Inversión total,ROI,Ganancia promedio,Decisión prom. ms,Composición,Pesos
5,Ensemble,Call,5,100,500,0.506,300,5000,0.060000,0.60,0.018335,"minimax, bayesian, markov","0.5, 0.3, 0.2"
2,Bayesian,Call,5,100,500,0.486,30,5000,0.006000,0.06,0.003959,,
1,Minimax,Call,5,100,500,0.480,10,5000,0.002000,0.02,0.006474,,
3,Markov,Call,5,100,500,0.462,-170,5000,-0.034000,-0.34,0.003888,,
0,Random,Call,5,100,500,0.278,-1515,5945,-0.254836,-3.03,0.003557,,
4,TDLearning,Call,5,100,500,0.000,-2500,2500,-1.000000,-5.00,0.004475,,


In [4]:
display(Markdown(render_spanish_conclusions(results)))


## Conclusiones según los resultados observados

La evaluación usó 5 semilla(s) y 100 mano(s) por semilla contra el baseline pasivo Call, con ROI corregido sobre la inversión real agregada.
El mejor resultado por ROI fue Ensemble: ROI 0.060000, win rate 0.506 y ganancia promedio 0.600 fichas por mano.
El ensemble calibrado usa minimax, bayesian, markov con pesos minimax=0.5, bayesian=0.3, markov=0.2; obtuvo ROI 0.060000, win rate 0.506 y ganancia promedio 0.600.
Caveat: la calibración se hizo con una grilla pequeña y semillas fijas frente a Call, Random y Markov; estos resultados son reproducibles, pero no reemplazan una validación estadística con más manos, más oponentes e intervalos de confianza.

## Fuente de verdad

| Capacidad | Módulo |
|---|---|
| Estado del juego, acciones legales, ciclo de apuestas y resultados de manos | `poker_ai.engine` |
| Agentes Random, Call, Minimax, Bayesian, Markov, TD, EnsembleNoTD y EnsembleWithTD | `poker_ai.agents` |
| Agregación de evaluación y ROI basado en inversión real | `poker_ai.evaluation` |

Este notebook intencionalmente no contiene definiciones centrales duplicadas; las pruebas automatizadas protegen ese contrato.